In [2]:
import requests
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

#czesc-1

url = "https://restcountries.com/v3.1/all"
response = requests.get(url)
data = response.json()

rows = []

for country in data:
    name = country.get("name", {}).get("common")
    
    capital = country.get("capital")
    capital = capital[0] if isinstance(capital, list) else None
    
    region = country.get("region")
    subregion = country.get("subregion")
    population = country.get("population")
    area = country.get("area")
    
    currencies = country.get("currencies")
    if currencies:
        currency = list(currencies.keys())[0]
    else:
        currency = None

    rows.append([
        name, capital, region, subregion,
        population, area, currency
    ])

df = pd.DataFrame(rows, columns=[
    "nazwa", "stolica", "region", "subregion",
    "populacja", "powierzchnia", "waluta"
])

print("HEAD:")
print(df.head())

print("\nSHAPE:")
print(df.shape)

print("\nDTYPES:")
print(df.dtypes)

conn = sqlite3.connect("kraje_swiata.db")
df.to_sql("kraje", conn, if_exists="replace", index=False)

print("\n1. Łączna populacja świata:")
print(pd.read_sql_query("""
SELECT SUM(populacja) AS total_population
FROM kraje
""", conn))


print("\n2. Top 10 krajów wg populacji:")
print(pd.read_sql_query("""
SELECT nazwa, populacja
FROM kraje
ORDER BY populacja DESC
LIMIT 10
""", conn))


print("\n3. Regiony — liczba krajów + średnia populacja:")
print(pd.read_sql_query("""
SELECT 
    region,
    COUNT(*) AS liczba_krajow,
    AVG(populacja) AS srednia_populacja
FROM kraje
GROUP BY region
ORDER BY srednia_populacja DESC
""", conn))


print("\n4. Kraje większe niż Polska (~312 679 km²):")
print(pd.read_sql_query("""
SELECT nazwa, powierzchnia
FROM kraje
WHERE powierzchnia > 312679
ORDER BY powierzchnia DESC
""", conn))


print("\n5. Największa gęstość zaludnienia:")
print(pd.read_sql_query("""
SELECT 
    nazwa,
    populacja,
    powierzchnia,
    (populacja * 1.0 / powierzchnia) AS gestosc
FROM kraje
WHERE powierzchnia IS NOT NULL AND powierzchnia > 0
ORDER BY gestosc DESC
LIMIT 1
""", conn))

region_pop = pd.read_sql_query("""
SELECT 
    region,
    SUM(populacja) AS total_population
FROM kraje
GROUP BY region
ORDER BY total_population DESC
""", conn)

plt.figure(figsize=(10,6))
plt.bar(region_pop["region"], region_pop["total_population"])
plt.xticks(rotation=45)
plt.title("Łączna populacja według regionów")
plt.xlabel("Region")
plt.ylabel("Populacja")
plt.show()

conn.close()

AttributeError: 'str' object has no attribute 'get'